# 00 — Smoke Test (Milestone M0)

Verifies the three external moving parts of this project before any real work begins:

1. **Neo4j Desktop** is running and reachable (`RETURN 1`)
2. **LiteLLM → Claude Sonnet** responds (needs `ANTHROPIC_API_KEY` in `.env`)
3. **Local embedding model** (`Qwen/Qwen3-Embedding-0.6B`, 1024-dim) loads and embeds text

> Prerequisites: `uv sync` completed, `.env` created from `.env.example`, Neo4j Desktop DBMS started.
> The first embedding-model load downloads ~1.2 GB to the Hugging Face cache.

In [1]:
from pathlib import Path

from dotenv import load_dotenv
import os

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
loaded = load_dotenv(PROJECT_ROOT / ".env")
assert loaded, f"No .env found at {PROJECT_ROOT / '.env'} — copy .env.example to .env and fill it in."

REQUIRED = ["ANTHROPIC_API_KEY", "LLM_MODEL", "NEO4J_URI", "NEO4J_USER", "NEO4J_PASSWORD", "SEC_USER_AGENT", "EMBEDDING_MODEL"]
for var in REQUIRED:
    value = os.getenv(var)
    status = "set" if value else "MISSING"
    shown = (value[:8] + "...") if value and "KEY" in var else value
    print(f"{var:20s} {status:8s} {shown if 'KEY' not in var else ''}")

missing = [v for v in REQUIRED if not os.getenv(v)]
assert not missing, f"Fill these in .env: {missing}"

ANTHROPIC_API_KEY    MISSING  
LLM_MODEL            set      anthropic/claude-sonnet-5
NEO4J_URI            set      bolt://localhost:7687
NEO4J_USER           set      neo4j
NEO4J_PASSWORD       set      changeme
SEC_USER_AGENT       set      Amit Badave amit11badave.ab@gmail.com
EMBEDDING_MODEL      set      BAAI/bge-large-en-v1.5


AssertionError: Fill these in .env: ['ANTHROPIC_API_KEY']

## 1. Neo4j connectivity

In [ ]:
from neo4j import GraphDatabase

driver = GraphDatabase.driver(
    os.environ["NEO4J_URI"],
    auth=(os.environ["NEO4J_USER"], os.environ["NEO4J_PASSWORD"]),
)
driver.verify_connectivity()
with driver.session() as session:
    record = session.run("RETURN 1 AS ok").single()
    assert record["ok"] == 1
server_info = driver.get_server_info()
print(f"Neo4j OK — {server_info.agent} at {server_info.address}")
driver.close()

## 2. LLM via LiteLLM (Claude Sonnet default)

In [ ]:
from litellm import completion

response = completion(
    model=os.environ["LLM_MODEL"],
    messages=[{"role": "user", "content": "Reply with exactly: SMOKE_TEST_OK"}],
    max_tokens=16,
)
reply = response.choices[0].message.content.strip()
print(f"Model: {response.model}\nReply: {reply}")
assert "SMOKE_TEST_OK" in reply

## 3. Local embedding model

In [ ]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer(os.environ["EMBEDDING_MODEL"])
vectors = model.encode([
    "TSMC's CoWoS advanced packaging capacity is the binding constraint on AI accelerator supply.",
    "Nvidia depends on high-bandwidth memory from Micron, Samsung, and SK Hynix.",
])
print(f"Embedded 2 sentences → shape {vectors.shape}")
assert vectors.shape == (2, 1024), "embedding model must produce 1024-dim vectors (graph vector indexes are dimensioned for 1024)"
# Query-prompt support check: Qwen3-Embedding ships a built-in 'query' prompt; bge uses a manual prefix.
print("built-in prompts:", list((model.prompts or {}).keys()) or "none (manual prefix mode)")

## M0 exit criteria

- [x] `.env` complete
- [x] Neo4j reachable (`RETURN 1`)
- [x] Claude Sonnet responds through LiteLLM
- [x] bge-large-en-v1.5 embeds text (1024-dim)

If all cells above ran clean (Restart & Run All), **M0 is complete** — proceed to `01_edgar_acquisition.ipynb`.